# 3.01 · MiBici model serving with FastAPI

**Phase 3 (deployment)** following the [Cookiecutter Data Science](https://cookiecutter-data-science.drivendata.org/using-the-template/) convention: `PHASE.NOTEBOOK-initials-description`. It runs after `2.05`, which registered the champions and replayed the forecast of one night.

- **Input:**
  - the prediction service running (`make serve`) with the champions exported from the Registry (`make export`: `models/mibici-<target>/` and `models/model_card.json`)
  - the MLflow server (`make mlflow`), for the batch forecast it is compared with
  - the feature table and the station profiles
- **Output:** nothing is written. The notebook checks the service: its models, its schema, the forecast of a whole night against the batch forecast, and its answer to invalid requests.

**Why.** The batch forecast of `2.05` writes a file every night. A service lets other systems ask for a forecast when they need it: the routing tool of the operations team, or a dashboard that recomputes the plan after a late data update. It only helps if it answers with the same models and the same numbers as the batch job, and if it rejects the rows that break the data contract instead of forecasting from them.

**Requisites:** `make mlflow`, `make export` and `make serve` (in that order, the last two in their own terminals or in the background).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import time

os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"  # keep the outputs readable
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import pandas as pd
import requests

from team2_mlops2026.features import FEATURES_DIR, STATION, TARGETS, TIME, forecast_rows
from team2_mlops2026.modeling.predict import NET_FLOW, join_profiles, load_history, predict
from team2_mlops2026.modeling.registry import CHAMPION
from team2_mlops2026.modeling.train import load_profiles
from team2_mlops2026.serving.schemas import MAX_ROWS, to_payload

pd.set_option("display.max_colwidth", 90)

2026-11-18 12:03:46.024 | INFO     | team2_mlops2026.config:<module>:12 - PROJ_ROOT path is: /Users/xaviermini/code/tec/mlops/finalProyecto/MLOps_Team2


## Parameters

The night is the one replayed in `2.05`: the forecast made on Tuesday 8 September 2026 for Wednesday 9 September, a regular weekday of the last test month.

In [3]:
SERVICE_URL = os.getenv("SERVICE_URL", "http://127.0.0.1:8000")  # make serve
NIGHT = pd.Timestamp("2026-09-09")  # the day forecast
TIMEOUT = 60  # seconds
CARD_TAGS = [
    "model",
    "algorithm",
    "training_window",
    "test_peak_wape",
    "backtest_peak_wape",
    "success_criterion",
    "peak_wape_improvement",
    "features_digest",
    "git_commit",
]

## 1. The service and the models it serves

`GET /health` says whether the models loaded, and `GET /model` returns the model card that `make export` wrote: for each target, the registered version that held the `champion` alias, its versioned URI, its source run and the tags the Registry keeps (metrics, input digests, git commit).

In [4]:
health = requests.get(f"{SERVICE_URL}/health", timeout=TIMEOUT)
print(f"GET /health -> {health.status_code} {health.json()}")

card = requests.get(f"{SERVICE_URL}/model", timeout=TIMEOUT).json()
print(f"Exported at {card['exported_at']} from commit {card['git_commit'][:7]}")
pd.DataFrame(
    {
        target: {
            "uri": entry["uri"],
            "alias": entry["alias"],
            "run_id": entry["run_id"],
            **{tag: entry["tags"].get(tag) for tag in CARD_TAGS},
        }
        for target, entry in card["models"].items()
    }
)

GET /health -> 200 {'status': 'ok', 'models_loaded': True, 'source': 'local'}
Exported at 2026-11-18T16:47:42+00:00 from commit 1db2b3f


,departures,arrivals
uri,models:/mibici-departures/1,models:/mibici-arrivals/1
alias,champion,champion
run_id,0524ecdd8541472a9f16070b7e3412cf,cd78b63838fb491ab138e321cfa76031
model,lgbm_tweedie,lgbm_tweedie
algorithm,LGBMRegressor,LGBMRegressor
training_window,2023-01 to 2025-10 (exclusive),2023-01 to 2025-10 (exclusive)
test_peak_wape,0.5198,0.4831
backtest_peak_wape,0.5198 ± 0.0421,0.4832 ± 0.0393
success_criterion,passed,passed
peak_wape_improvement,0.1538,0.1511


**Result → action.** The service loaded both models from `models/` (`source: local`), so it answers without the MLflow server.
- It serves **`models:/mibici-departures/1`** and **`models:/mibici-arrivals/1`**, the `lgbm_tweedie` versions that hold `champion` in the Registry (`2.05`).
- The card carries what identifies them:
  - the source runs and the training window (2023-01 to 2025-09);
  - the test peak-hour WAPE (0.520 and 0.483) and the backtest mean ± std;
  - the success criterion (passed, 15.4% and 15.1% below `historical_average`);
  - the features digest and the git commit of the training code (`f247380`).
- The export itself was made on 18 November 2026 from commit `1db2b3f`.

→ A client can check which version answered without access to MLflow, and the README lists the same URIs. To serve another version, move the alias, run `make export` and restart the service; the card then shows the new version.

## 2. The request and response schema

The service publishes its OpenAPI schema at `/openapi.json` (Swagger UI at `/docs`), and a copy is kept in `docs/api/openapi.json`. The tables below read it: each field of a request row, with its type, whether it is required and its bounds.

In [5]:
schema = requests.get(f"{SERVICE_URL}/openapi.json", timeout=TIMEOUT).json()
components = schema["components"]["schemas"]
BOUNDS = ("minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum", "minItems", "maxItems")


def fields(name: str) -> pd.DataFrame:
    spec = components[name]
    required = set(spec.get("required", []))
    rows = []
    for field, prop in spec["properties"].items():
        options = prop.get("anyOf", [prop])
        types = [o.get("type") or o.get("$ref", "").split("/")[-1] for o in options]
        bounds = {k: v for o in [prop, *options] for k, v in o.items() if k in BOUNDS}
        rows.append(
            {
                "field": field,
                "type": " | ".join(types),
                "required": field in required,
                "bounds": bounds or "",
                "description": prop.get("description", ""),
            }
        )
    return pd.DataFrame(rows).set_index("field")


endpoints = pd.DataFrame(
    [
        {
            "endpoint": f"{method.upper()} {path}",
            "summary": op.get("summary", ""),
            "responses": ", ".join(op["responses"]),
        }
        for path, methods in schema["paths"].items()
        for method, op in methods.items()
    ]
).set_index("endpoint")
display(endpoints)
display(fields("PredictRequest"))
fields("FeatureRow")

,summary,responses
endpoint,,
POST /predict,Forecast the departures and arrivals of station-hours,"200, 400, 500, 503, 422"
GET /health,Whether the models are loaded,"200, 503"
GET /model,The model card of the served models,"200, 503"


,type,required,bounds,description
field,,,,
rows,array,True,"{'maxItems': 25000, 'minItems': 1}","Station-hours to forecast, 1 to 25,000"


,type,required,bounds,description
field,,,,
station_id,integer,True,{'exclusiveMinimum': 0.0},MiBici station id
hour_start,string,True,,"Hour to forecast, local time without time zone"
departures_lag_2d,number | null,True,{'minimum': 0.0},"Trips (null if unknown, e.g. before the station opened)"
departures_lag_7d,number | null,True,{'minimum': 0.0},"Trips (null if unknown, e.g. before the station opened)"
departures_same_hour_4w,number | null,True,{'minimum': 0.0},"Trips (null if unknown, e.g. before the station opened)"
departures_mean_7d,number | null,True,{'minimum': 0.0},"Trips (null if unknown, e.g. before the station opened)"
arrivals_lag_2d,number | null,True,{'minimum': 0.0},"Trips (null if unknown, e.g. before the station opened)"
arrivals_lag_7d,number | null,True,{'minimum': 0.0},"Trips (null if unknown, e.g. before the station opened)"
arrivals_same_hour_4w,number | null,True,{'minimum': 0.0},"Trips (null if unknown, e.g. before the station opened)"


In [6]:
fields("Prediction")

,type,required,bounds,description
field,,,,
station_id,integer,True,,
hour_start,string,True,,
departures,number,True,,Expected departures in the hour
arrivals,number,True,,Expected arrivals in the hour
net_flow,number,True,,arrivals - departures: < 0 empties the station


**Result → action.** The schema is the data contract of `features.forecast_rows`, checked on every row:
- **23 fields per row.** The station and the hour, the 8 demand lags, the weekday, the 4 cyclical terms and the 3 station attributes are required. The 5 profile scores are optional.
- **`null` is allowed** where the feature table has NaN (a lag before the station opened, a station missing from the nomenclature), since the pipeline imputes it.
- **The bounds:** non-negative lags, weekday 0-6, sin/cos in [-1, 1], coordinates inside Guadalajara, capacity > 0. Unknown fields are rejected, so a target sent by mistake never reaches the model.
- **A batch** has 1 to 25,000 rows: a whole night (~11,200) with room to grow.
- **The response** gives the expected departures, arrivals and net flow of each row, plus the version of each model.

→ The schema is versioned in `docs/api/openapi.json`, with an example request and response. A test fails if the service and that copy drift apart.

## 3. A whole night through the service

The rows of the night are built exactly as the batch forecast builds them: `features.forecast_rows` from the history before 9 September, plus the station's profile scores of the week (`predict.join_profiles`). `to_payload` turns them into the JSON body of `POST /predict`.

The same night is then forecast in batch with `predict()`, which loads the models from the Registry by alias (`models:/mibici-<target>@champion`). Both must give the same numbers for every station-hour.

In [7]:
scores, _ = load_profiles(FEATURES_DIR)
rows = join_profiles(forecast_rows(load_history(FEATURES_DIR, NIGHT), NIGHT), scores)
payload = to_payload(rows)

start = time.perf_counter()
response = requests.post(f"{SERVICE_URL}/predict", json=payload, timeout=TIMEOUT)
elapsed = time.perf_counter() - start
response.raise_for_status()
body = response.json()
served = pd.DataFrame(body["predictions"])
served[TIME] = pd.to_datetime(served[TIME]).astype(rows[TIME].dtype)

print(
    f"POST /predict: {len(rows):,} rows ({rows[STATION].nunique()} stations x 24 h), "
    f"{len(response.request.body) / 1e6:.1f} MB, answered in {elapsed:.2f} s"
)
print("Models:", ", ".join(m["uri"] for m in body["models"].values()))
print("Example row:", payload["rows"][0])

POST /predict: 11,208 rows (467 stations x 24 h), 7.4 MB, answered in 0.28 s
Models: models:/mibici-departures/1, models:/mibici-arrivals/1
Example row: {'station_id': 2, 'hour_start': '2026-09-09T00:00:00', 'departures_lag_2d': 0.0, 'departures_lag_7d': 0.0, 'departures_same_hour_4w': 0.0, 'departures_mean_7d': 2.2142856121063232, 'arrivals_lag_2d': 0.0, 'arrivals_lag_7d': 0.0, 'arrivals_same_hour_4w': 0.0, 'arrivals_mean_7d': 2.202380895614624, 'weekday': 2, 'hour_sin': 0.0, 'hour_cos': 1.0, 'month_sin': -0.8660254037844384, 'month_cos': -0.5000000000000004, 'latitude': 20.666378, 'longitude': -103.34882, 'capacity': 15.0, 'pc1': -0.0008786644320935011, 'pc2': -0.007662998978048563, 'pc3': -0.0016884561628103256, 'pc4': -0.005561758764088154, 'pc5': 0.007891055196523666}


In [8]:
batch = predict(NIGHT, dst=None, alias=CHAMPION)
compared = batch.predictions.merge(
    served, on=[STATION, TIME], suffixes=("_batch", "_api"), validate="one_to_one"
)
print(
    f"Batch: {len(batch.predictions):,} rows, service: {len(served):,} rows, "
    f"matched: {len(compared):,}"
)
pd.DataFrame(
    {
        column: {
            "batch total": compared[f"{column}_batch"].sum(),
            "service total": compared[f"{column}_api"].sum(),
            "max abs difference": (compared[f"{column}_batch"] - compared[f"{column}_api"])
            .abs()
            .max(),
        }
        for column in [*TARGETS, NET_FLOW]
    }
)

/Users/xaviermini/code/tec/mlops/finalProyecto/MLOps_Team2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-11-18 12:03:48.351 | SUCCESS  | team2_mlops2026.modeling.predict:predict:132 - Forecast of 11,208 station-hours (467 stations) from 2026-09-09 00:00:00 with mibici-departures v1, mibici-arrivals v1
Batch: 11,208 rows, service: 11,208 rows, matched: 11,208


,departures,arrivals,net_flow
batch total,14833.767995,14792.921043,-40.846952
service total,14833.767995,14792.921043,-40.846952
max abs difference,0.000000,0.000000,0.000000


In [9]:
daily = served.groupby(STATION)[[*TARGETS, NET_FLOW]].sum()
daily = daily.assign(capacity=rows.groupby(STATION)["capacity"].first())
daily["net_flow / capacity"] = daily[NET_FLOW] / daily["capacity"]
print("Stations that will empty the most (daily net flow from the service):")
display(daily.nsmallest(5, NET_FLOW).round(1))
print("Stations that will fill the most:")
daily.nlargest(5, NET_FLOW).round(1)

Stations that will empty the most (daily net flow from the service):


,departures,arrivals,net_flow,capacity,net_flow / capacity
station_id,,,,,
194,140.5,44.0,-96.5,19.0,-5.1
288,83.0,55.6,-27.4,31.0,-0.9
271,225.1,200.0,-25.1,31.0,-0.8
196,69.1,47.3,-21.8,23.0,-0.9
11,180.4,163.4,-16.9,23.0,-0.7


Stations that will fill the most:


,departures,arrivals,net_flow,capacity,net_flow / capacity
station_id,,,,,
35,223.8,263.6,39.8,15.0,2.7
34,108.9,148.5,39.5,15.0,2.6
51,304.0,335.6,31.6,39.0,0.8
260,124.2,153.2,29.0,15.0,1.9
384,43.4,70.7,27.3,23.0,1.2


**Result → action.**
- **Same numbers as the batch forecast.** The service answered the 11,208 rows of the night (467 stations × 24 h), a 7.4 MB request, in under half a second.
  - Every station-hour matches the batch forecast of `predict()` exactly: the largest difference is 0 in departures, arrivals and net flow.
  - The totals are those of `2.05`: 14,834 departures and 14,793 arrivals.
  - The rows travel as JSON and come back with the feature table's dtypes (`schemas.ROW_DTYPES`), so the models see the same inputs.
- **Same action list.** The stations with the largest daily imbalance are those of `2.05`:
  - station 194 will lose 96.5 bikes against 19 docks, five times its capacity;
  - stations 35 and 34 will gain about 40 bikes against 15 docks each.

→ Any consumer of the nightly file can use the service instead. The batch job stays the reference, and `test_the_api_forecasts_equal_the_batch_forecast` keeps both equal.

## 4. Invalid requests

The service validates every row against the data contract before forecasting. Each case below changes one field of a valid row (or the size of the batch) and shows the status and the message the client gets.

In [10]:
row = payload["rows"][0]
cases = {
    "valid row": [row],
    "without profile scores (pc1-pc5)": [{k: v for k, v in row.items() if not k.startswith("pc")}],
    "weekday out of range": [row | {"weekday": 9}],
    "weekday of another day": [row | {"weekday": (row["weekday"] + 1) % 7}],
    "negative lag": [row | {"departures_lag_2d": -3}],
    "hour with a time zone": [row | {"hour_start": "2026-09-09T00:00:00Z"}],
    "station in Mexico City": [row | {"latitude": 19.43, "longitude": -99.13}],
    "target sent as a feature": [row | {"departures": 5}],
    "capacity missing": [{k: v for k, v in row.items() if k != "capacity"}],
    "empty batch": [],
    f"{MAX_ROWS + 1:,} rows": [row] * (MAX_ROWS + 1),
}


def message(answer: requests.Response) -> str:
    detail = answer.json().get("detail")
    if detail is None:
        return f"{len(answer.json()['predictions'])} prediction(s)"
    if isinstance(detail, str):
        return detail
    return "; ".join(f"{'.'.join(map(str, e['loc'][2:]))}: {e['msg']}" for e in detail)


results = []
for case, case_rows in cases.items():
    answer = requests.post(f"{SERVICE_URL}/predict", json={"rows": case_rows}, timeout=TIMEOUT)
    results.append({"case": case, "status": answer.status_code, "message": message(answer)})
pd.DataFrame(results).set_index("case")

,status,message
case,,
valid row,200,1 prediction(s)
without profile scores (pc1-pc5),200,1 prediction(s)
weekday out of range,422,0.weekday: Input should be less than or equal to 6
weekday of another day,422,"0: Value error, weekday 3 does not match hour_start 2026-09-09, a weekday 2 (Monday = 0)"
negative lag,422,0.departures_lag_2d: Input should be greater than or equal to 0
hour with a time zone,422,"0.hour_start: Value error, must be local time without a time zone, as in the feature t..."
station in Mexico City,422,0.latitude: Input should be greater than or equal to 20.5; 0.longitude: Input should b...
target sent as a feature,422,0.departures: Extra inputs are not permitted
capacity missing,422,0.capacity: Field required


**Result → action.** Every invalid request is rejected before it reaches the model, with a status the client can act on:
- **422** when a row breaks the contract. The message names the row, the field and the reason: a weekday out of range or of another day, a negative lag, an hour with a time zone, a station outside Guadalajara, a target sent as a feature, a missing field.
- **400** for an empty batch or one of more than 25,000 rows.
- **200** for a row without profile scores. The pipeline treats it as a station-week without a profile (the average profile plus the indicator), as in training.

Two statuses do not show here: 503 while the models are not loaded, and 500 if the forecast itself fails. `tests/test_serving.py` covers both. → A client that gets 422 builds its rows wrong; one that gets 503 has to wait for `make export` and a restart.

## Findings and next steps

1. **Service.** `make serve` starts a FastAPI service that loads the champions exported from the Registry (`models:/mibici-departures/1` and `models:/mibici-arrivals/1`, alias `champion`). It needs no MLflow server. `GET /health`, `GET /model` and the Swagger UI at `/docs` describe it.
2. **Same numbers as the batch.** For the night of 9 September 2026, the 11,208 forecasts of `POST /predict` equal those of the batch forecast exactly, and the service answered in under half a second.
3. **Contract enforced.** Invalid rows get 422 with the field and the reason, and a batch out of range gets 400. A row without profile scores is still forecast.
4. **Documented.** The OpenAPI schema and an example request and response are in `docs/api/`, and the README records the URIs and versions served.

Next steps:
- Package the service with the exported models in a Docker image and publish it on DockerHub (#40).
- Run the service and the pipeline in a clean environment and compare their outputs with these references (#41).
- Optional: serve the next day's forecast straight from the feature table (`POST /forecast`) and log the served predictions for the drift monitoring (#47).